In [11]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder

# --- FILNAMN ---
INPUT_FILE = 'KDDTrain+.txt' 

def train_and_save_all():
    print("🚀 Startar träning av de tre jämförelsemodellerna...")

    # 1. Inläsning av NSL-KDD
    columns = [
        'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 
        'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
        'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
        'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
        'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
        'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
        'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
        'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
        'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
        'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'class', 'difficulty_level'
    ]

    if not os.path.exists(INPUT_FILE):
        print(f"❌ Fel: Hittade inte {INPUT_FILE}")
        return

    df = pd.read_csv(INPUT_FILE, names=columns, header=None)

    # 2. EDA & Preprocessing
    df['bad_packet'] = df['class'].apply(lambda x: 1 if x != 'normal' else 0)
    le = LabelEncoder()
    df['protocol_type'] = le.fit_transform(df['protocol_type'])
    
    # Valda särdrag (viktigt att backend använder exakt samma ordning!)
    features = ['duration', 'src_bytes', 'dst_bytes', 'count', 'diff_srv_rate', 'protocol_type']
    X = df[features].fillna(0)
    y = df['bad_packet']

    # 3. Normalisering
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    joblib.dump(scaler, 'scaler.pkl')
    print("✅ Scaler sparad (scaler.pkl)")

    # 4. Träning och Sparning av varje enskild modell
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

    # Vi skapar en ordbok med våra modeller och deras önskade filnamn
    model_configs = {
        "Random Forest": (RandomForestClassifier(n_estimators=100, class_weight='balanced'), 'rf_model.pkl'),
        "Logistic Regression": (LogisticRegression(max_iter=1000, class_weight='balanced'), 'lr_model.pkl'),
        "Gradient Boosting": (GradientBoostingClassifier(n_estimators=100), 'gb_model.pkl')
    }

    for name, (model, filename) in model_configs.items():
        print(f"🧠 Tränar {name}...")
        model.fit(X_train, y_train)
        joblib.dump(model, filename)
        print(f"✅ {name} sparad som {filename}")

    # 5. Spara skarp testdata för simuleringen (oskalad för att backend ska kunna skala den själv)
    test_raw_X, _, test_raw_y, _ = train_test_split(df[features], y, test_size=0.2, random_state=42)
    test_export = pd.concat([test_raw_X, test_raw_y], axis=1)
    test_export.to_csv('skarp_test_data.txt', index=False, quoting=3, escapechar=' ')
    print("💾 Skarp testdata sparad.")

if __name__ == "__main__":
    train_and_save_all()

🚀 Startar träning av de tre jämförelsemodellerna...
✅ Scaler sparad (scaler.pkl)
🧠 Tränar Random Forest...
✅ Random Forest sparad som rf_model.pkl
🧠 Tränar Logistic Regression...
✅ Logistic Regression sparad som lr_model.pkl
🧠 Tränar Gradient Boosting...
✅ Gradient Boosting sparad som gb_model.pkl
💾 Skarp testdata sparad.
